# 01c Global LSTM EDA And Data Preparation

Prepare pooled/global LSTM artifacts that train on all eligible ticker series at once. No scaler is fit here; all scaling remains inside strict train/refit splits.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
for candidate in [cwd, cwd / "forecasting", cwd.parent, cwd.parent / "forecasting"]:
    if (candidate / "src" / "stock_forecast").exists():
        PROJECT_DIR = candidate
        break
else:
    raise RuntimeError("Cannot locate forecasting project directory with src/stock_forecast")

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ARTIFACT_DIR = PROJECT_DIR / "artifacts"
DATA_DIR = ARTIFACT_DIR / "data"
REPORTS_DIR = ARTIFACT_DIR / "reports"
GLOBAL_LSTM_DATA_DIR = DATA_DIR / "global_lstm"
for path in [GLOBAL_LSTM_DATA_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_DIR = {PROJECT_DIR}")

PROJECT_DIR = /home/sapce/forecasting_stock_prices/forecasting


In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

from stock_forecast.artifacts import load_json, load_table, save_json, save_table
from stock_forecast.lstm_features import sequence_diagnostics

pd.set_option("display.max_columns", 180)

## Build Global Feature Sets

In [3]:
HORIZONS = [
    {"name": "week", "horizon": 5},
    {"name": "month", "horizon": 21},
]
RECOMMENDED_LOOKBACKS = [20, 40, 60, 90, 126]
LEVEL_LIKE_FEATURES = {
    "log_close",
    "dollar_volume",
    "lstm_dollar_volume_log",
}


def add_ticker_one_hot(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    out = frame.copy()
    tickers = sorted(out["ticker"].astype(str).unique())
    cols = []
    for ticker in tickers:
        col = f"ticker_onehot_{ticker}"
        out[col] = (out["ticker"].astype(str) == ticker).astype(float)
        cols.append(col)
    return out, cols


def stationary_feature_columns(feature_cols: list[str], ticker_onehot_cols: list[str]) -> list[str]:
    return [
        col
        for col in feature_cols
        if col not in LEVEL_LIKE_FEATURES and not col.endswith("_log")
    ] + ticker_onehot_cols


global_artifacts = []
global_summaries = {}

for spec in HORIZONS:
    horizon_name = spec["name"]
    horizon = int(spec["horizon"])
    lstm_dir = DATA_DIR / "lstm" / "horizons" / horizon_name
    model_df = load_table(lstm_dir / "model_dataset.parquet")
    model_df["date"] = pd.to_datetime(model_df["date"])
    payload = load_json(lstm_dir / "feature_columns.json")
    target_col = payload["target_column"]
    model_df, ticker_onehot_cols = add_ticker_one_hot(model_df)

    global_all = list(payload["feature_columns"]) + ticker_onehot_cols
    global_stationary = stationary_feature_columns(payload["feature_columns"], ticker_onehot_cols)
    feature_sets = {
        "global_all": global_all,
        "global_stationary": global_stationary,
    }

    out_dir = GLOBAL_LSTM_DATA_DIR / "horizons" / horizon_name
    model_path = save_table(model_df, out_dir / "model_dataset.parquet")
    global_payload = {
        **payload,
        "feature_source": "01c_global_lstm_eda",
        "ticker_onehot_columns": ticker_onehot_cols,
        "global_feature_sets": feature_sets,
        "primary_global_feature_set": "global_stationary",
        "scaler_policy": "fit median imputer and RobustScaler inside each strict global train/refit split only",
        "target_normalization_policy": "global protocol tunes global vs per_ticker target normalization; default per_ticker",
        "recommended_lookbacks": RECOMMENDED_LOOKBACKS,
    }
    save_json(global_payload, out_dir / "feature_columns.json")

    diagnostics = {
        feature_set_name: sequence_diagnostics(
            model_df,
            feature_cols=cols,
            target_col=target_col,
            lookbacks=RECOMMENDED_LOOKBACKS,
        )
        for feature_set_name, cols in feature_sets.items()
    }
    diagnostics["horizon_name"] = horizon_name
    diagnostics["horizon"] = horizon
    diagnostics["target_column"] = target_col
    save_json(diagnostics, out_dir / "global_sequence_diagnostics.json")

    rows_by_ticker = pd.DataFrame(diagnostics["global_stationary"]["rows_by_ticker"])
    total_sequences_60 = int(rows_by_ticker["sequences_lookback_60"].sum())
    global_artifacts.append({
        "horizon_name": horizon_name,
        "horizon": horizon,
        "rows_model_dataset": int(len(model_df)),
        "tickers": int(model_df["ticker"].nunique()),
        "global_all_features": len(global_all),
        "global_stationary_features": len(global_stationary),
        "sequences_lookback_60": total_sequences_60,
        "target_column": target_col,
        "model_dataset": str(model_path.relative_to(PROJECT_DIR)),
        "feature_columns": str((out_dir / "feature_columns.json").relative_to(PROJECT_DIR)),
        "sequence_diagnostics": str((out_dir / "global_sequence_diagnostics.json").relative_to(PROJECT_DIR)),
    })
    global_summaries[horizon_name] = diagnostics

summary = {
    "source_notebook": "01c_global_lstm_eda.ipynb",
    "source_lstm_notebook": "01b_lstm_eda.ipynb",
    "horizons": HORIZONS,
    "feature_sets": ["global_all", "global_stationary"],
    "artifacts": global_artifacts,
}
save_json(summary, REPORTS_DIR / "global_lstm_eda_summary.json")
display(pd.DataFrame(global_artifacts))

,horizon_name,horizon,rows_model_dataset,tickers,global_all_features,global_stationary_features,sequences_lookback_60,target_column,model_dataset,feature_columns,sequence_diagnostics
0,week,5,19407,7,167,164,18994,target_return_5_next_open,artifacts/data/global_lstm/horizons/week/model...,artifacts/data/global_lstm/horizons/week/featu...,artifacts/data/global_lstm/horizons/week/globa...
1,month,21,19295,7,167,164,18882,target_return_21_next_open,artifacts/data/global_lstm/horizons/month/mode...,artifacts/data/global_lstm/horizons/month/feat...,artifacts/data/global_lstm/horizons/month/glob...


## Sequence Feasibility

In [4]:
for horizon_name, diagnostics in global_summaries.items():
    print(f"=== {horizon_name} / global_stationary ===")
    display(pd.DataFrame(diagnostics["global_stationary"]["rows_by_ticker"]))
    display(pd.Series(diagnostics["global_stationary"]["target_distribution"], name=horizon_name))

=== week / global_stationary ===


,ticker,rows,date_start,date_end,max_feasible_lookback,sequences_lookback_20,sequences_lookback_40,sequences_lookback_60,sequences_lookback_90,sequences_lookback_126
0,CBOM,2758,2015-11-09,2026-06-07,126,2739,2719,2699,2669,2633
1,MBNK,537,2024-09-05,2026-06-07,126,518,498,478,448,412
2,SBER,4629,2007-11-28,2026-06-07,126,4610,4590,4570,4540,4504
3,SBERP,4629,2007-11-28,2026-06-07,126,4610,4590,4570,4540,4504
4,SVCB,636,2024-04-27,2026-06-07,126,617,597,577,547,511
5,T,1633,2020-03-13,2026-06-07,126,1614,1594,1574,1544,1508
6,VTBR,4585,2008-02-01,2026-06-07,126,4566,4546,4526,4496,4460


count     19407.000000
mean          0.000654
std           0.057441
min          -0.826040
p25          -0.021111
median        0.001016
p75           0.023865
max           0.584459
Name: week, dtype: float64

=== month / global_stationary ===


,ticker,rows,date_start,date_end,max_feasible_lookback,sequences_lookback_20,sequences_lookback_40,sequences_lookback_60,sequences_lookback_90,sequences_lookback_126
0,CBOM,2742,2015-11-09,2026-05-22,126,2723,2703,2683,2653,2617
1,MBNK,521,2024-09-05,2026-05-22,126,502,482,462,432,396
2,SBER,4613,2007-11-28,2026-05-22,126,4594,4574,4554,4524,4488
3,SBERP,4613,2007-11-28,2026-05-22,126,4594,4574,4554,4524,4488
4,SVCB,620,2024-04-27,2026-05-22,126,601,581,561,531,495
5,T,1617,2020-03-13,2026-05-22,126,1598,1578,1558,1528,1492
6,VTBR,4569,2008-02-01,2026-05-22,126,4550,4530,4510,4480,4444


count     19295.000000
mean          0.002706
std           0.115971
min          -1.129788
p25          -0.044760
median        0.006187
p75           0.058624
max           0.687158
Name: month, dtype: float64